# Dish Clustering — Modelling Playground

Standalone experimentation notebook for exploring dish clustering — the
sandbox that comes *before* committing to a fixed pipeline, not a wrapper
around one. Everything here is self-contained so weights, algorithms, and
features can be changed freely.

- Tune the feature-block weights (canonical / subgroup / group / method)
- Sweep `k` for KMeans and inspect silhouette / Calinski-Harabasz / Davies-Bouldin scores
- Compare KMeans against Agglomerative, GaussianMixture, Spectral, and DBSCAN
- Visualise clusters with t-SNE and PCA layouts (Plotly, interactive)
- Inspect cluster membership and auto-generated labels

## Setup

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import plotly.express as px
import plotly.graph_objects as go

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, normalize
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)

In [ ]:
def find_repo_root(start: Path, marker: str = "ml") -> Path:
    for path in [start, *start.parents]:
        if (path / marker).is_dir():
            return path
    raise FileNotFoundError(f"no parent of {start} contains a '{marker}' directory")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
ML_DIR = REPO_ROOT / "ml"
sys.path.insert(0, str(ML_DIR))

import ingredient_taxonomy as tax # claude generated food group hierarchy. Working well for now, could change to a more established food categorization FOODEX2

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR    = REPO_ROOT / "data"
INPUT_FILE  = DATA_DIR / "recipes.json"
OUTPUT_FILE = DATA_DIR / "clusters.json"

# ── Config ───────────────────────────────────────────────────────────────────
N_CLUSTERS      = 8
TOP_CONNECTIONS = 4
SIM_THRESHOLD   = 0.12   # minimum cosine similarity to include a connection
TSNE_PERPLEXITY = 10
RANDOM_STATE    = 42

# Weighting of the four feature blocks:
W_CANONICAL  = 0.5   # fine-grained TF-IDF over canonical ingredient tokens
W_SUBGROUP   = 0.2
W_GROUP      = 0.15
W_METHOD     = 0.15

ALL_GROUPS = list(tax.HIERARCHY.keys())

# ── Cluster label hints - claude generated, could be improved ─────────────────
CLUSTER_THEME_MAP = {
    frozenset(["beef", "potato", "flour"]):            "Hearty meat & pastry",
    frozenset(["lamb", "onion", "carrot"]):            "Slow-cooked lamb broths",
    frozenset(["pasta", "egg", "cured pork"]):         "Roman pasta classics",
    frozenset(["pasta", "tomato", "beef"]):            "Meaty pasta sauces",
    frozenset(["cream", "sugar", "egg"]):              "Rich cream desserts",
    frozenset(["flour", "olive oil", "tomato"]):       "Baked doughs & breads",
    frozenset(["ricotta", "flour", "whisky"]):         "Sicilian sweets",
    frozenset(["pork", "oats", "black pepper"]):       "Cured & spiced meats",
    frozenset(["risotto rice", "parmesan", "saffron"]):"Northern Italian rice",
    frozenset(["stale bread", "tomato", "olive oil"]): "Tuscan bread dishes",
    frozenset(["potato", "leek", "butter"]):           "British comfort food",
    frozenset(["oats", "barley", "lamb"]):             "Scottish & Northern broths",
    frozenset(["flour", "butter", "sugar"]):           "Baked goods & puddings",
    frozenset(["white fish", "potato", "cream"]):      "Fish & seafood dishes",
}

In [ ]:
def load_recipes(path: Path) -> list[dict]:
    with open(path) as f:
        return json.load(f)


# ── Feature engineering ──────────────────────────────────────────────────────

def recipe_to_ingredient_lines(recipe: dict) -> list[str]:
    """Extract raw ingredient item strings from a recipe."""
    return [ing["item"] for ing in recipe.get("ingredients", [])]


def recipe_to_canonical_doc(recipe: dict) -> str:
    """
    Return a space-joined string of canonical ingredient tokens for TF-IDF.
    Uses ingredient_taxonomy.normalize() — much richer than the old alias dict.
    Unrecognised ingredients are included as-is (lowercased) so no signal is lost.
    Also appends any recipe tags as extra tokens.
    """
    lines = recipe_to_ingredient_lines(recipe)
    tokens = []
    for line in lines:
        canon = tax.normalize(line)
        tokens.append(canon if canon is not None else line.lower())
    tags = recipe.get("tags", [])
    return " ".join(tokens + tags)


def recipe_to_subgroup_doc(recipe: dict) -> str:
    """
    Space-joined string of subgroup tokens (one per recognised ingredient).
    Tokens are repeated proportionally to ingredient count via Counter.
    """
    lines = recipe_to_ingredient_lines(recipe)
    counts = tax.featurize(lines, "subgroup")
    # repeat each subgroup token by its count so TF-IDF sees frequency
    return " ".join(
        token.replace(" ", "_").replace("&", "and")
        for token, cnt in counts.items()
        for _ in range(cnt)
    )


def recipe_group_flags(recipe: dict) -> list[str]:
    """
    Return the list of top-level hierarchy groups present in this recipe.
    Used for one-hot group presence features.
    """
    lines = recipe_to_ingredient_lines(recipe)
    group_counts = tax.featurize(lines, "group")
    return [g for g in ALL_GROUPS if g in group_counts]


def build_feature_matrix(recipes: list[dict]):
    """
    Combine four weighted feature blocks:
      1. TF-IDF of canonical ingredient tokens          (W_CANONICAL)
      2. TF-IDF of subgroup tokens                      (W_SUBGROUP)
      3. Multi-hot group presence (one column per group) (W_GROUP)
      4. One-hot cooking method                          (W_METHOD)

    All blocks are L2-normalised before weighting so weights are comparable.
    Returns the combined L2-normalised sparse matrix.
    """
    canonical_docs  = [recipe_to_canonical_doc(r)  for r in recipes]
    subgroup_docs   = [recipe_to_subgroup_doc(r)   for r in recipes]
    methods         = [[r.get("cooking_method", "unknown")] for r in recipes]

    # ── Block 1: canonical TF-IDF ────────────────────────────────────────────
    tfidf_canon = TfidfVectorizer(min_df=1, ngram_range=(1, 2), sublinear_tf=True)
    X_canon = tfidf_canon.fit_transform(canonical_docs)   # sparse

    # ── Block 2: subgroup TF-IDF ─────────────────────────────────────────────
    tfidf_sub = TfidfVectorizer(min_df=1, ngram_range=(1, 1), sublinear_tf=True)
    X_sub = tfidf_sub.fit_transform(subgroup_docs)         # sparse

    # ── Block 3: group multi-hot ─────────────────────────────────────────────
    # Build a binary matrix: rows = recipes, cols = ALL_GROUPS
    group_matrix = np.zeros((len(recipes), len(ALL_GROUPS)), dtype=np.float32)
    for i, recipe in enumerate(recipes):
        present = set(recipe_group_flags(recipe))
        for j, grp in enumerate(ALL_GROUPS):
            if grp in present:
                group_matrix[i, j] = 1.0
    X_group = sp.csr_matrix(group_matrix)

    # ── Block 4: cooking method one-hot ──────────────────────────────────────
    ohe = OneHotEncoder(
        categories=[tax.COOKING_METHODS + ["unknown"]],
        sparse_output=True,
        handle_unknown="ignore",
    )
    X_method = ohe.fit_transform(methods)

    # ── L2-normalise each block independently, then weight and combine ────────
    def safe_norm(X):
        return normalize(X, norm="l2")

    X_combined = sp.hstack([
        safe_norm(X_canon)  * W_CANONICAL,
        safe_norm(X_sub)    * W_SUBGROUP,
        safe_norm(X_group)  * W_GROUP,
        safe_norm(X_method) * W_METHOD,
    ])

    # Final L2 normalise rows for cosine similarity
    X_norm = normalize(X_combined, norm="l2")
    return X_norm, tfidf_canon, tfidf_sub, ohe


# ── Clustering ───────────────────────────────────────────────────────────────

def run_kmeans(X, n_clusters: int, random_state: int = RANDOM_STATE):
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    labels = km.fit_predict(X)
    return labels, km


def run_tsne(X, perplexity: int = TSNE_PERPLEXITY, random_state: int = RANDOM_STATE):
    """Return 2D t-SNE coordinates as a list of [x, y] pairs."""
    X_dense = X.toarray() if sp.issparse(X) else X
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        max_iter=2000,
        learning_rate="auto",
        init="pca",
    )
    coords = tsne.fit_transform(X_dense)
    # Normalise to [-1, 1] range for easy frontend use
    for i in range(2):
        mn, mx = coords[:, i].min(), coords[:, i].max()
        coords[:, i] = 2 * (coords[:, i] - mn) / (mx - mn + 1e-9) - 1
    return coords.tolist()


# ── Similarity connections ───────────────────────────────────────────────────

def build_connections(X, recipe_ids: list[str], top_k: int, threshold: float):
    """
    For each dish, find the top_k most similar other dishes above threshold.
    Returns a list of { source, target, similarity } dicts.
    """
    X_dense = X.toarray() if sp.issparse(X) else X
    sim_matrix = cosine_similarity(X_dense)  # (n, n)
    connections = []
    seen = set()
    for i, src_id in enumerate(recipe_ids):
        row = sim_matrix[i].copy()
        row[i] = 0  # exclude self
        top_indices = np.argsort(row)[::-1][:top_k]
        for j in top_indices:
            score = float(row[j])
            if score < threshold:
                continue
            # deduplicate undirected edges
            edge_key = tuple(sorted([src_id, recipe_ids[j]]))
            if edge_key in seen:
                continue
            seen.add(edge_key)
            connections.append({
                "source": src_id,
                "target": recipe_ids[j],
                "similarity": round(score, 4),
            })
    # sort by similarity descending
    connections.sort(key=lambda x: x["similarity"], reverse=True)
    return connections


# ── Cluster label inference ──────────────────────────────────────────────────

def infer_cluster_label(cluster_id: int, members: list[dict]) -> str:
    """
    Derive a readable cluster label using the taxonomy hierarchy.

    Strategy:
      1. Aggregate canonical counts across all recipes in the cluster.
      2. Check CLUSTER_THEME_MAP for a ≥2-keyword match.
      3. If no match, fall back to the dominant group name + top 2 canonicals.
    """
    canonical_counts: Counter = Counter()
    group_counts: Counter = Counter()

    for r in members:
        lines = recipe_to_ingredient_lines(r)
        canonical_counts.update(tax.featurize(lines, "canonical"))
        group_counts.update(tax.featurize(lines, "group"))

    # Exclude near-universal seasonings from label generation
    LABEL_STOPWORDS = {"salt", "black pepper", "olive oil", "butter", "egg",
                       "flour", "water", "sugar", "stock"}

    top_canonicals = [
        k for k, _ in canonical_counts.most_common(10)
        if k not in LABEL_STOPWORDS
    ][:5]

    keyword_set = frozenset(top_canonicals[:3])
    for theme_keys, label in CLUSTER_THEME_MAP.items():
        if len(theme_keys & keyword_set) >= 2:
            return label

    # Fallback: dominant group + top 2 distinctive canonicals
    dominant_group = group_counts.most_common(1)[0][0].title() if group_counts else ""
    top2 = " · ".join(top_canonicals[:2]).title()
    if dominant_group and top2:
        return f"{dominant_group}: {top2}"
    if top2:
        return top2
    return f"Cluster {cluster_id + 1}"

## Load data

In [ ]:
recipes = load_recipes(INPUT_FILE)
recipe_ids = [r["id"] for r in recipes]
recipe_names = [r["name"] for r in recipes]
print(f"{len(recipes)} dishes loaded from {INPUT_FILE}")
# recipes[0]

## Feature engineering

`build_feature_matrix` combines four L2-normalised, weighted blocks:

1. **Canonical** — TF-IDF over normalised ingredient tokens (`W_CANONICAL`)
2. **Subgroup** — TF-IDF over taxonomy subgroup tokens (`W_SUBGROUP`)
3. **Group** — multi-hot presence of top-level taxonomy groups (`W_GROUP`)
4. **Method** — one-hot cooking method (`W_METHOD`)

The weights are plain globals defined above in this notebook — mutate them
before calling `build_feature_matrix` to experiment.

In [ ]:
X, tfidf_canon, tfidf_sub, ohe = build_feature_matrix(recipes)
X_dense = X.toarray()
print(f"Feature matrix: {X.shape[0]} dishes x {X.shape[1]} features "
      f"(density={X.nnz / (X.shape[0] * X.shape[1]):.3f})")

## Clustering playground

Rows of `X` are already L2-normalised, so Euclidean distance on `X_dense` is
monotonic with cosine distance — plain (Euclidean) sklearn metrics are safe
to use directly below.

In [6]:
def evaluate_clustering(X_dense, labels, name=""):
    """Print + return silhouette / Calinski-Harabasz / Davies-Bouldin for a labelling.
    Points labelled -1 (DBSCAN noise) are excluded from scoring."""
    labels = np.asarray(labels)
    mask = labels != -1
    n_found = len(set(labels[mask]))
    if n_found < 2:
        print(f"{name:30s}  fewer than 2 clusters found — skipping metrics")
        return None
    sil = silhouette_score(X_dense[mask], labels[mask])
    ch = calinski_harabasz_score(X_dense[mask], labels[mask])
    db = davies_bouldin_score(X_dense[mask], labels[mask])
    n_noise = int((~mask).sum())
    print(f"{name:30s}  silhouette={sil:.3f}  calinski_harabasz={ch:7.1f}  "
          f"davies_bouldin={db:.3f}  n_clusters={n_found}  n_noise={n_noise}")
    return {"name": name, "silhouette": sil, "calinski_harabasz": ch,
            "davies_bouldin": db, "n_clusters": n_found, "n_noise": n_noise}

### KMeans — sweep `k`

In [ ]:
k_results = []
for k in range(2, 15):
    labels, _ = run_kmeans(X, k)
    r = evaluate_clustering(X_dense, labels, f"KMeans k={k}")
    if r:
        r["k"] = k
        k_results.append(r)

df_k = pd.DataFrame(k_results)
df_k

In [8]:
fig = go.Figure()
fig.add_scatter(x=df_k["k"], y=df_k["silhouette"], name="silhouette", mode="lines+markers")
fig.update_layout(
    title="Silhouette score vs. k (higher is better)",
    xaxis_title="k", yaxis_title="silhouette score", width=700, height=400,
)
fig.show()

### Compare algorithms at a fixed `k`

Swap in / tune any of these — e.g. adjust `eps` for DBSCAN using the
k-distance plot below, or try `linkage="complete"` for Agglomerative.

In [11]:
# N_CLUSTERS is already defined above — override here to experiment, e.g. N_CLUSTERS = 6

algorithms = {
    "KMeans": KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20),
    "Agglomerative (avg/cosine)": AgglomerativeClustering(
        n_clusters=N_CLUSTERS, linkage="average", metric="cosine"
    ),
    "GaussianMixture": GaussianMixture(n_components=N_CLUSTERS, random_state=RANDOM_STATE),
    "Spectral": SpectralClustering(
        n_clusters=N_CLUSTERS, affinity="nearest_neighbors", n_neighbors=10,
        random_state=RANDOM_STATE,
    )
}

cluster_results = {}
metric_rows = []
for name, model in algorithms.items():
    labels = model.fit_predict(X_dense)
    cluster_results[name] = np.asarray(labels)
    r = evaluate_clustering(X_dense, labels, name)
    if r:
        metric_rows.append(r)

df_algos = pd.DataFrame(metric_rows).set_index("name")
df_algos

NameError: name 'X_dense' is not defined

DBSCAN's `eps` needs tuning to the data — use the k-distance plot below (look
for the "elbow") to pick a reasonable value, then re-run the cell above.

In [10]:
k_nn = 4
nbrs = NearestNeighbors(n_neighbors=k_nn, metric="cosine").fit(X_dense)
distances, _ = nbrs.kneighbors(X_dense)
k_distances = np.sort(distances[:, -1])

fig = px.line(y=k_distances, title=f"{k_nn}-distance plot (DBSCAN eps tuning)")
fig.update_layout(xaxis_title="points, sorted", yaxis_title=f"distance to {k_nn}th nearest neighbor",
                   width=700, height=400)
fig.show()

## Dimensionality reduction for visualisation

`run_tsne` returns 2D coordinates normalised to `[-1, 1]` (handy if these ever
feed a frontend). PCA is added alongside as a faster, deterministic alternative.

In [ ]:
coords_tsne = np.array(run_tsne(X, perplexity=min(TSNE_PERPLEXITY, len(recipes) - 1)))

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords_pca = pca.fit_transform(X_dense)
print(f"PCA explained variance ratio (2 components): {pca.explained_variance_ratio_.sum():.3f}")

In [12]:
def plot_clusters(coords, labels, title):
    fig = px.scatter(
        x=coords[:, 0], y=coords[:, 1],
        color=[str(l) for l in labels],
        hover_name=recipe_names,
        title=title,
        labels={"color": "cluster"},
        category_orders={"color": sorted({str(l) for l in labels})},
    )
    fig.update_traces(marker=dict(size=10, line=dict(width=1, color="white")))
    fig.update_layout(width=750, height=550)
    return fig

plot_clusters(coords_tsne, cluster_results["KMeans"], "KMeans clusters — t-SNE layout").show()

In [13]:
# Flip between algorithms / layouts here
ALGO = "GaussianMixture"
LAYOUT = coords_tsne  # or coords_pca

plot_clusters(LAYOUT, cluster_results[ALGO], f"{ALGO} clusters").show()

## Inspect cluster membership

In [ ]:
def summarize_clusters(recipes, labels):
    members = {}
    for r, l in zip(recipes, labels):
        members.setdefault(int(l), []).append(r)
    for cid in sorted(members):
        m = members[cid]
        label = infer_cluster_label(cid, m) if cid != -1 else "Noise (DBSCAN)"
        names = ", ".join(r["name"] for r in m)
        print(f"[{cid}] {label}  ({len(m)} dishes)")
        print(f"    {names}\n")

summarize_clusters(recipes, cluster_results["KMeans"])

## Dish-to-dish similarity connections

For each dish, find the `top_k` most similar other dishes above a similarity
`threshold`. Sanity-checking that a chosen feature weighting produces sensible
nearest-neighbour pairs.

In [ ]:
connections = build_connections(X, recipe_ids, TOP_CONNECTIONS, SIM_THRESHOLD)
print(f"{len(connections)} edges found\n")

id_to_name = {r["id"]: r["name"] for r in recipes}
for c in connections[:15]:
    print(f"  {id_to_name[c['source']]}  <->  {id_to_name[c['target']]}  (sim={c['similarity']})")